# Analisis de estrategias SMC

**Parte 1** — Grid search (BTCUSDT 1m, 100 estrategias)

**Parte 2** — Backtest anual: BTCUSDT / ETHUSDT / XRPUSDT / SOLUSDT × 1d / 4h / 1h

> Prerequisitos:
> ```bash
> python -m backtests.strategy_selector   # genera data/results.csv
> python -m backtests.annual_backtest     # genera data/annual_results.csv y annual_detail.csv
> ```

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

DATA_DIR = os.path.join("..", "data")

---
## PARTE 1 — Grid search BTCUSDT 1m (100 estrategias)

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "results.csv"))
df_valid = df[df["trades"] >= 3].copy()
df_top10 = df_valid.head(10).copy()

print(f"Estrategias totales: {len(df)}")
print(f"Con >= 3 trades:     {len(df_valid)}")
df_top10

### 1 · Top 10 por PnL total

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

labels = [
    f"sw={r.swing_window} hold={r.max_hold}\nfvg={'si' if r.require_fvg else 'no'} choch={'si' if r.use_choch_filter else 'no'}"
    for _, r in df_top10.iterrows()
]
colors = ["steelblue" if v >= 0 else "tomato" for v in df_top10["total_pnl"]]
bars = ax.bar(range(len(df_top10)), df_top10["total_pnl"] * 100, color=colors)
ax.set_xticks(range(len(df_top10)))
ax.set_xticklabels(labels, fontsize=8)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f%%"))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Top 10 estrategias — PnL total (%)", fontsize=13)
ax.set_ylabel("PnL (%)")

for bar, val in zip(bars, df_top10["total_pnl"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (0.001 if val >= 0 else -0.003),
            f"{val*100:+.2f}%", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

### 2 · PnL vs Winrate

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(df_valid["winrate"] * 100, df_valid["total_pnl"] * 100,
                c=df_valid["trades"], cmap="viridis", s=60, alpha=0.75, edgecolors="none")
plt.colorbar(sc, ax=ax, label="N de trades")
ax.scatter(df_top10["winrate"] * 100, df_top10["total_pnl"] * 100,
           marker="*", s=200, color="red", label="Top 10", zorder=5)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.axvline(50, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("Winrate (%)")
ax.set_ylabel("PnL total (%)")
ax.set_title("PnL vs Winrate — todas las estrategias validas", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

### 3 · Heatmap: swing_window vs max_hold

In [ ]:
pivot = df_valid.pivot_table(values="total_pnl", index="swing_window",
                              columns="max_hold", aggfunc="mean") * 100
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("max_hold")
ax.set_ylabel("swing_window")
ax.set_title("PnL medio (%) por swing_window x max_hold", fontsize=13)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i,j]:.2f}%", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="PnL medio (%)")
plt.tight_layout()
plt.show()

### 4 · Efecto de require_fvg y use_choch_filter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, title, labels in zip(
    axes,
    ["require_fvg", "use_choch_filter"],
    ["Efecto de require_fvg", "Efecto de use_choch_filter"],
    [["Sin FVG", "Con FVG"], ["Sin filtro CHoCH", "Con filtro CHoCH"]],
):
    means = df_valid.groupby(col)["total_pnl"].mean() * 100
    colors = ["steelblue" if v >= 0 else "tomato" for v in means]
    bars = ax.bar(labels, means.values, color=colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2f%%"))
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("PnL medio (%)")
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + (0.0005 if val >= 0 else -0.002),
                f"{val:+.3f}%", ha="center", va="bottom", fontsize=10)
plt.suptitle("Impacto de los filtros booleanos en PnL medio", fontsize=13)
plt.tight_layout()
plt.show()

---
## PARTE 2 — Backtest anual: 4 pares × 3 timeframes (1d / 4h / 1h)

In [ ]:
df_annual = pd.read_csv(os.path.join(DATA_DIR, "annual_results.csv"))
df_detail = pd.read_csv(os.path.join(DATA_DIR, "annual_detail.csv"))

print(f"Combinaciones par x timeframe: {len(df_annual)}")
df_annual

### 5 · PnL anual por par y timeframe

In [ ]:
pairs = df_annual["pair"].unique()
timeframes = df_annual["timeframe"].unique()
x = np.arange(len(pairs))
width = 0.25
tf_colors = {"1d": "steelblue", "4h": "seagreen", "1h": "darkorange"}

fig, ax = plt.subplots(figsize=(11, 5))

for i, tf in enumerate(timeframes):
    subset = df_annual[df_annual["timeframe"] == tf].set_index("pair")
    vals = [subset.loc[p, "total_pnl"] * 100 if p in subset.index else 0 for p in pairs]
    offset = (i - len(timeframes) / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=tf,
                  color=tf_colors.get(tf, "gray"), alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + (0.05 if val >= 0 else -0.15),
                f"{val:+.1f}%", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(pairs, fontsize=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.1f%%"))
ax.set_ylabel("PnL (%)")
ax.set_title("PnL anual por par y timeframe — mejor estrategia por combinacion", fontsize=13)
ax.legend(title="Timeframe")
plt.tight_layout()
plt.show()

### 6 · Winrate y Profit Factor por par × timeframe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, ylabel, hline, hline_label in [
    (axes[0], "winrate", "Winrate (%)", 0.5, "50%"),
    (axes[1], "profit_factor", "Profit Factor", 1.0, "PF=1"),
]:
    for i, tf in enumerate(timeframes):
        subset = df_annual[df_annual["timeframe"] == tf].set_index("pair")
        scale = 100 if metric == "winrate" else 1
        vals = [subset.loc[p, metric] * scale if p in subset.index else 0 for p in pairs]
        offset = (i - len(timeframes) / 2 + 0.5) * width
        ax.bar(x + offset, vals, width, label=tf,
               color=tf_colors.get(tf, "gray"), alpha=0.85)
    ax.axhline(hline * (100 if metric == "winrate" else 1),
               color="gray", linewidth=1, linestyle="--", label=hline_label)
    ax.set_xticks(x)
    ax.set_xticklabels(pairs)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + " por par y timeframe", fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle("Calidad de las mejores estrategias por par × timeframe", fontsize=13)
plt.tight_layout()
plt.show()

### 7 · Heatmap: PnL medio de los 100 strategies — par × timeframe

In [ ]:
df_det_valid = df_detail[df_detail["trades"] >= 3]
pivot_pt = df_det_valid.pivot_table(
    values="total_pnl", index="pair", columns="timeframe", aggfunc="mean"
) * 100

# Reordenar timeframes
tf_order = [t for t in ["1d", "4h", "1h"] if t in pivot_pt.columns]
pivot_pt = pivot_pt[tf_order]

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot_pt.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(pivot_pt.columns)))
ax.set_xticklabels(pivot_pt.columns, fontsize=10)
ax.set_yticks(range(len(pivot_pt.index)))
ax.set_yticklabels(pivot_pt.index, fontsize=10)
ax.set_xlabel("Timeframe")
ax.set_ylabel("Par")
ax.set_title("PnL medio (%) — 100 estrategias por par × timeframe", fontsize=12)
for i in range(len(pivot_pt.index)):
    for j in range(len(pivot_pt.columns)):
        val = pivot_pt.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}%", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="PnL medio (%)")
plt.tight_layout()
plt.show()

### 8 · Distribucion de PnL por timeframe (todas las estrategias validas)

In [ ]:
fig, axes = plt.subplots(1, len(tf_order), figsize=(14, 4), sharey=True)

for ax, tf in zip(axes, tf_order):
    subset = df_det_valid[df_det_valid["timeframe"] == tf]["total_pnl"] * 100
    color = "steelblue" if subset.median() >= 0 else "tomato"
    ax.hist(subset, bins=25, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.axvline(subset.median(), color="orange", linewidth=1.5,
               label=f"median={subset.median():.2f}%")
    ax.set_title(f"Timeframe: {tf}", fontsize=11)
    ax.set_xlabel("PnL (%)")
    ax.legend(fontsize=8)

axes[0].set_ylabel("N estrategias")
plt.suptitle("Distribucion de PnL — todos los pares y estrategias validas", fontsize=13)
plt.tight_layout()
plt.show()

### 9 · Heatmap swing_window × max_hold por par y timeframe

In [ ]:
pairs_list = df_annual["pair"].unique()

fig, axes = plt.subplots(len(tf_order), len(pairs_list),
                          figsize=(16, 3.5 * len(tf_order)))

for row, tf in enumerate(tf_order):
    for col, pair in enumerate(pairs_list):
        ax = axes[row][col]
        subset = df_det_valid[
            (df_det_valid["pair"] == pair) & (df_det_valid["timeframe"] == tf)
        ]
        if subset.empty:
            ax.axis("off")
            continue
        pivot = subset.pivot_table(
            values="total_pnl", index="swing_window", columns="max_hold", aggfunc="mean"
        ) * 100
        im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, fontsize=6)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=6)
        ax.set_title(f"{pair} · {tf}", fontsize=9)
        if col == 0:
            ax.set_ylabel("swing", fontsize=7)
        if row == len(tf_order) - 1:
            ax.set_xlabel("hold", fontsize=7)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                            fontsize=5, color="black")

plt.suptitle("PnL medio (%) por swing_window × max_hold — par × timeframe", fontsize=13)
plt.tight_layout()
plt.show()